# 01 Logistic Regression: Predict Penguin Species

This notebook uses Logistic Regression to predict `species`. It is a stable and interpretable multi-class baseline model.


In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TEST_SIZE = 0.25

from sklearn.linear_model import LogisticRegression


## 1. Load Data


In [ ]:
candidate_paths = [
    Path('../../data/ml_data/penguins.csv'),
    Path('../data/ml_data/penguins.csv'),
    Path('/workspace/data/ml_data/penguins.csv'),
    Path('/Users/nancy/Desktop/cancer-predictor/data/ml_data/penguins.csv'),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Cannot find penguins.csv. Please check data/ml_data/penguins.csv')

penguins = pd.read_csv(data_path)
print('Data path:', data_path)
print('Data shape:', penguins.shape)
penguins.head()


## 2. Prepare Training and Test Sets

The three modeling notebooks use the same feature columns and `random_state=42`. They also stratify by `species` so the test-set class proportions stay consistent.


In [ ]:
target_col = 'species'
drop_cols = ['Unnamed: 0']
feature_cols = [col for col in penguins.columns if col not in drop_cols + [target_col]]

X = penguins[feature_cols].copy()
y = penguins[target_col].copy()

numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'year']
categorical_features = ['island', 'sex']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Test-set class distribution:')
print(y_test.value_counts().sort_index())


## 3. Build the Preprocessing and Logistic Regression Pipeline


In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

logistic_kwargs = {'max_iter': 2000}
if 'multi_class' in inspect.signature(LogisticRegression).parameters:
    logistic_kwargs['multi_class'] = 'auto'

logistic_regression = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(**logistic_kwargs)),
])

logistic_regression.fit(X_train, y_train)
print(logistic_regression)


## 4. Predict and Evaluate


In [ ]:
y_pred = logistic_regression.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification report:')
print(classification_report(y_test, y_pred))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Logistic Regression Confusion Matrix')
plt.xticks(rotation=30, ha='right')
plt.show()

pd.DataFrame({'actual': y_test.values, 'predicted': y_pred}, index=y_test.index).head(12)


## 5. Summary

Logistic Regression provides a clear linear baseline. If accuracy and macro F1 are already high, the penguin species boundaries are fairly clear in these measurement features.
